In [1]:
from pynq import MMIO

gpio_cmd = MMIO(0x41200000, 0x1000)
gpio_status = MMIO(0x41210000, 0x1000)

status = gpio_status.read(0x00)

print(f"Status = 0x{status : 08X}")

Status = 0x 0000000


In [2]:
from pynq import MMIO
import time

gpio_cmd = MMIO(0x41200000, 0x1000)
gpio_status = MMIO(0x41210000, 0x1000)

# Registre de commande :
# bits 0-7 : data_in
# bit 8    : data_last
# bit 9    : send_toggle

command = 0xAA | (1 << 8) | (1 << 9)

gpio_cmd.write(0x00, command)

time.sleep(0.01)

status = gpio_status.read(0x00)

print(f"Status = 0x{status:08X}")
print(f"Frame ready = {(status >> 20) & 1}")
print(f"Frame size  = {(status >> 8) & 0xFFF}")
print(f"Send seen   = {(status >> 22) & 1}")
print(f"Overflow    = {(status >> 21) & 1}")

Status = 0x00500100
Frame ready = 1
Frame size  = 1
Send seen   = 1
Overflow    = 0


In [3]:
import time

# Commande actuelle : 0x000003AA
command = gpio_cmd.read(0x00)

# Demander la lecture à l'adresse 0
command &= ~(0x7FF << 11)
command ^= (1 << 22)

gpio_cmd.write(0x00, command)

# Attendre la confirmation du bridge
for _ in range(1000):
    status = gpio_status.read(0x00)

    read_toggle = (command >> 22) & 1
    read_seen = (status >> 24) & 1

    if read_seen == read_toggle:
        break

    time.sleep(0.001)
else:
    raise TimeoutError("Le FPGA n'a pas confirmé la lecture")

# Extraire l'octet lu
data = status & 0xFF

print(f"Status    = 0x{status:08X}")
print(f"Read data = 0x{data:02X}")

assert data == 0xAA, "La BRAM ne contient pas la valeur attendue"

print("Lecture BRAM réussie !")

Status    = 0x015001AA
Read data = 0xAA
Lecture BRAM réussie !


In [4]:
from pynq import MMIO
import time

# ==========================================
# AXI GPIO
# ==========================================

GPIO_CMD_BASE = 0x41200000
GPIO_STATUS_BASE = 0x41210000

gpio_cmd = MMIO(GPIO_CMD_BASE, 0x1000)
gpio_status = MMIO(GPIO_STATUS_BASE, 0x1000)

CMD_OFFSET = 0x00
STATUS_OFFSET = 0x00

SEND_TOGGLE = 9
ACK_TOGGLE = 10
READ_TOGGLE = 22

FRAME_READY = 20
OVERFLOW = 21
SEND_SEEN = 22
ACK_SEEN = 23
READ_SEEN = 24

# ==========================================
# Trame de test : en-tete Ethernet II
# ==========================================

ethernet_header = bytes.fromhex(
    "00 E0 4C 68 27 F6 "
    "76 D6 D6 82 15 66 "
    "08 00"
)

assert len(ethernet_header) == 14

# ==========================================
# Utilitaires
# ==========================================

command = gpio_cmd.read(CMD_OFFSET)

def bit(value, position):
    return (value >> position) & 1


def wait_status(predicate, description, timeout=2.0):
    deadline = time.monotonic() + timeout

    while time.monotonic() < deadline:
        status = gpio_status.read(STATUS_OFFSET)

        if predicate(status):
            return status

        time.sleep(0.0001)

    raise TimeoutError(
        f"{description} : pas de confirmation du FPGA"
    )


def write_command():
    gpio_cmd.write(CMD_OFFSET, command)


def toggle_command(cmd_bit, seen_bit):
    global command

    command ^= (1 << cmd_bit)
    expected = bit(command, cmd_bit)

    write_command()

    return wait_status(
        lambda s: bit(s, seen_bit) == expected,
        f"Commande bit {cmd_bit}"
    )


def acknowledge():
    status = toggle_command(ACK_TOGGLE, ACK_SEEN)

    status = wait_status(
        lambda s: bit(s, FRAME_READY) == 0,
        "Liberation du buffer"
    )

    return status


def send_byte(value, is_last=False):
    global command

    # Remplacer les bits de donnees et data_last
    command &= ~0x1FF
    command |= (value & 0xFF)

    if is_last:
        command |= (1 << 8)

    return toggle_command(SEND_TOGGLE, SEND_SEEN)


def read_byte(address):
    global command

    if not 0 <= address < 2048:
        raise ValueError("Adresse BRAM invalide")

    # Remplacer read_addr (bits 11 a 21)
    command &= ~(0x7FF << 11)
    command |= address << 11

    status = toggle_command(READ_TOGGLE, READ_SEEN)

    return status & 0xFF


# ==========================================
# Verification de l'etat initial
# ==========================================

status = gpio_status.read(STATUS_OFFSET)

# Ne pas commencer si une ancienne commande
# n'a pas encore ete acquittee.
for cmd_bit, seen_bit in (
    (SEND_TOGGLE, SEND_SEEN),
    (ACK_TOGGLE, ACK_SEEN),
    (READ_TOGGLE, READ_SEEN),
):
    assert bit(command, cmd_bit) == bit(status, seen_bit), (
        "Commande GPIO encore en attente"
    )

# La trame AA du test precedent est peut-etre
# encore dans READY. L'acquitter si necessaire.
if bit(status, FRAME_READY):
    acknowledge()

# ==========================================
# TEST : Ecriture des 14 octets
# ==========================================

print("Envoi de l'en-tete Ethernet...")

for index, value in enumerate(ethernet_header):
    last = index == len(ethernet_header) - 1

    send_byte(value, is_last=last)

    print(f"TX [{index:02d}] = 0x{value:02X}")

# ==========================================
# Verification des statuts
# ==========================================

status = wait_status(
    lambda s: bit(s, FRAME_READY) == 1,
    "Trame complete"
)

frame_size = (status >> 8) & 0xFFF
overflow = bit(status, OVERFLOW)

print(f"\nFrame size : {frame_size}")
print(f"Overflow   : {overflow}")
print(f"Status     : 0x{status:08X}")

assert frame_size == 14, "Taille incorrecte"
assert overflow == 0, "Depassement memoire"

# ==========================================
# Lecture des 14 octets
# ==========================================

print("\nLecture de la BRAM...")

received = bytearray()

for address in range(len(ethernet_header)):
    value = read_byte(address)
    received.append(value)

    print(f"RX [{address:02d}] = 0x{value:02X}")

# ==========================================
# Comparaison
# ==========================================

assert bytes(received) == ethernet_header, (
    "Les octets relus sont differents"
)

print("\nEn-tete transmis :", ethernet_header.hex(" "))
print("En-tete recupere :", bytes(received).hex(" "))

print("\nTEST REUSSI : les 14 octets sont identiques !")

# Liberation pour le prochain test
acknowledge()


Envoi de l'en-tete Ethernet...
TX [00] = 0x00
TX [01] = 0xE0
TX [02] = 0x4C
TX [03] = 0x68
TX [04] = 0x27
TX [05] = 0xF6
TX [06] = 0x76
TX [07] = 0xD6
TX [08] = 0xD6
TX [09] = 0x82
TX [10] = 0x15
TX [11] = 0x66
TX [12] = 0x08
TX [13] = 0x00

Frame size : 14
Overflow   : 0
Status     : 0x01D00EAA

Lecture de la BRAM...
RX [00] = 0x00
RX [01] = 0xE0
RX [02] = 0x4C
RX [03] = 0x68
RX [04] = 0x27
RX [05] = 0xF6
RX [06] = 0x76
RX [07] = 0xD6
RX [08] = 0xD6
RX [09] = 0x82
RX [10] = 0x15
RX [11] = 0x66
RX [12] = 0x08
RX [13] = 0x00

En-tete transmis : 00 e0 4c 68 27 f6 76 d6 d6 82 15 66 08 00
En-tete recupere : 00 e0 4c 68 27 f6 76 d6 d6 82 15 66 08 00

TEST REUSSI : les 14 octets sont identiques !


20971520

In [1]:
from pynq import MMIO
import time

# AXI GPIO du Block Design Vivado
gpio_cmd = MMIO(0x41200000, 0x1000)
gpio_status = MMIO(0x41210000, 0x1000)
gpio_dst = MMIO(0x41220000, 0x1000)
gpio_src = MMIO(0x41230000, 0x1000)

# Configurer les GPIO en entrée/sortie
# Offset 0x04 : TRI canal 1
# Offset 0x0C : TRI canal 2
gpio_cmd.write(0x04, 0x00000000)

gpio_status.write(0x04, 0xFFFFFFFF)

for gpio in (gpio_dst, gpio_src):
    gpio.write(0x04, 0xFFFFFFFF)
    gpio.write(0x0C, 0xFFFFFFFF)

print("AXI GPIO initialisés")

AXI GPIO initialisés


In [2]:
def read_parser():
    dst_low = gpio_dst.read(0x00)
    dst_high = gpio_dst.read(0x08)

    src_low = gpio_src.read(0x00)
    src_high = gpio_src.read(0x08)

    print(f"dst_low  = 0x{dst_low:08X}")
    print(f"dst_high = 0x{dst_high:08X}")
    print(f"src_low  = 0x{src_low:08X}")
    print(f"src_high = 0x{src_high:08X}")

    print(f"header_done  = {(dst_high >> 16) & 1}")
    print(f"header_valid = {(dst_high >> 17) & 1}")
    print(f"is_ipv4      = {(dst_high >> 18) & 1}")
    print(f"is_arp       = {(dst_high >> 19) & 1}")

read_parser()

dst_low  = 0x00000000
dst_high = 0x00000000
src_low  = 0x00000000
src_high = 0x00000000
header_done  = 0
header_valid = 0
is_ipv4      = 0
is_arp       = 0


In [5]:
SEND_TOGGLE = 1 << 9
ACK_TOGGLE = 1 << 10

send_toggle = (gpio_status.read(0x00) >> 22) & 1
ack_toggle = (gpio_status.read(0x00) >> 23) & 1

def send_byte(value, last=False):
    global send_toggle

    send_toggle ^= 1

    command = (
        (value & 0xFF)
        | (int(last) << 8)
        | (send_toggle << 9)
        | (ack_toggle << 10)
    )

    gpio_cmd.write(0x00, command)

    for _ in range(10000):
        status = gpio_status.read(0x00)

        if ((status >> 22) & 1) == send_toggle:
            return

    raise TimeoutError("Le FPGA n'a pas acquitté l'octet")

print("Fonction send_byte() initialisée")

Fonction send_byte() initialisée


In [6]:
frame = bytes.fromhex(
    "00 E0 4C 68 27 F6 76 D6 D6 82 15 66 08 00"
)

for index, value in enumerate(frame):
    send_byte(value, last=(index == len(frame) - 1))

print(f"{len(frame)} octets envoyés")

14 octets envoyés


In [7]:
import time

time.sleep(0.01)

status = gpio_status.read(0x00)

print(f"Frame size  : {(status >> 8) & 0xFFF}")
print(f"Frame ready : {(status >> 20) & 1}")
print(f"Overflow    : {(status >> 21) & 1}")

read_parser()

Frame size  : 14
Frame ready : 1
Overflow    : 0
dst_low  = 0x4C6827F6
dst_high = 0x000700E0
src_low  = 0xD6821566
src_high = 0x080076D6
header_done  = 1
header_valid = 1
is_ipv4      = 1
is_arp       = 0


In [8]:
ack_toggle ^= 1

command = (
    (send_toggle << 9)
    | (ack_toggle << 10)
)

gpio_cmd.write(0x00, command)

for _ in range(10000):
    status = gpio_status.read(0x00)
    if ((status >> 23) & 1) == ack_toggle:
        break
else:
    raise TimeoutError("ACK non confirmé")

print(f"Frame ready : {(status >> 20) & 1}")
read_parser()

Frame ready : 0
dst_low  = 0x00000000
dst_high = 0x00000000
src_low  = 0x00000000
src_high = 0x00000000
header_done  = 0
header_valid = 0
is_ipv4      = 0
is_arp       = 0


In [9]:
arp_frame = bytes.fromhex(
    "FF FF FF FF FF FF 12 34 56 78 9A BC 08 06"
)

for index, value in enumerate(arp_frame):
    send_byte(value, last=(index == len(arp_frame) - 1))

time.sleep(0.01)

status = gpio_status.read(0x00)

print(f"Frame size  : {(status >> 8) & 0xFFF}")
print(f"Frame ready : {(status >> 20) & 1}")
print(f"Overflow    : {(status >> 21) & 1}")

read_parser()

Frame size  : 14
Frame ready : 1
Overflow    : 0
dst_low  = 0xFFFFFFFF
dst_high = 0x000BFFFF
src_low  = 0x56789ABC
src_high = 0x08061234
header_done  = 1
header_valid = 1
is_ipv4      = 0
is_arp       = 1


In [10]:
ack_toggle ^= 1

gpio_cmd.write(
    0x00,
    (send_toggle << 9) | (ack_toggle << 10)
)

for _ in range(10000):
    status = gpio_status.read(0x00)
    if ((status >> 23) & 1) == ack_toggle:
        break
else:
    raise TimeoutError("ACK non confirmé")

print("Frame ready:", (status >> 20) & 1)

Frame ready: 0


In [11]:
short_frame = bytes.fromhex(
    "00 E0 4C 68 27 F6 76 D6 D6 82"
)

for index, value in enumerate(short_frame):
    send_byte(value, last=(index == len(short_frame) - 1))

time.sleep(0.01)

status = gpio_status.read(0x00)

print("Frame size :", (status >> 8) & 0xFFF)
print("Frame ready:", (status >> 20) & 1)
print("Overflow   :", (status >> 21) & 1)

read_parser()

Frame size : 10
Frame ready: 1
Overflow   : 0
dst_low  = 0x4C6827F6
dst_high = 0x000100E0
src_low  = 0x76D6D682
src_high = 0x00000000
header_done  = 1
header_valid = 0
is_ipv4      = 0
is_arp       = 0


In [12]:
ack_toggle ^= 1

gpio_cmd.write(
    0x00,
    (send_toggle << 9) | (ack_toggle << 10)
)

for _ in range(10000):
    status = gpio_status.read(0x00)
    if ((status >> 23) & 1) == ack_toggle:
        break
else:
    raise TimeoutError("ACK non confirmé")

read_parser()

dst_low  = 0x00000000
dst_high = 0x00000000
src_low  = 0x00000000
src_high = 0x00000000
header_done  = 0
header_valid = 0
is_ipv4      = 0
is_arp       = 0


In [13]:
import time

BUFFER_SIZE = 2048

# En-tête Ethernet II : EtherType IPv4
header = bytes.fromhex(
    "00 E0 4C 68 27 F6 76 D6 D6 82 15 66 08 00"
)

# Trame de 2049 octets : dépassement d'un octet
overflow_frame = header + bytes(BUFFER_SIZE + 1 - len(header))

assert len(overflow_frame) == 2049

# Vérifier que le buffer est libre avant le test
status = gpio_status.read(0x00)
assert ((status >> 20) & 1) == 0, "Buffer occupé : effectuer ACK avant le test"

# Envoyer les octets via notre bridge AXI GPIO
for index, value in enumerate(overflow_frame):
    send_byte(
        value,
        last=(index == len(overflow_frame) - 1)
    )

time.sleep(0.01)

# Lire le statut final
status = gpio_status.read(0x00)

frame_size = (status >> 8) & 0xFFF
frame_ready = (status >> 20) & 1
overflow = (status >> 21) & 1

print(f"Octets envoyés : {len(overflow_frame)}")
print(f"Frame size     : {frame_size}")
print(f"Frame ready    : {frame_ready}")
print(f"Overflow       : {overflow}")

print("\n--- Parser Ethernet ---")
read_parser()

assert overflow == 1, "ÉCHEC : overflow non détecté"
print("\nPASS : débordement détecté par le FPGA")

Octets envoyés : 2049
Frame size     : 2048
Frame ready    : 1
Overflow       : 1

--- Parser Ethernet ---
dst_low  = 0x4C6827F6
dst_high = 0x000100E0
src_low  = 0xD6821566
src_high = 0x080076D6
header_done  = 1
header_valid = 0
is_ipv4      = 0
is_arp       = 0

PASS : débordement détecté par le FPGA


In [14]:
ack_toggle ^= 1

gpio_cmd.write(
    0x00,
    (send_toggle << 9) | (ack_toggle << 10)
)

for _ in range(10000):
    status = gpio_status.read(0x00)
    if ((status >> 23) & 1) == ack_toggle:
        break
else:
    raise TimeoutError("ACK non confirmé")

status = gpio_status.read(0x00)

print("Frame ready:", (status >> 20) & 1)
print("Overflow   :", (status >> 21) & 1)

read_parser()

Frame ready: 0
Overflow   : 0
dst_low  = 0x00000000
dst_high = 0x00000000
src_low  = 0x00000000
src_high = 0x00000000
header_done  = 0
header_valid = 0
is_ipv4      = 0
is_arp       = 0
